In [7]:
# Step 0: Setting up
!pip install requests pandas numpy matplotlib seaborn scipy

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import warnings
import os
 
warnings.filterwarnings("ignore")
 
# Output folder for all saved figures 
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
# Shared plot style 
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.facecolor": "white",
    "axes.facecolor": "#F9F9F9",
    "axes.grid": True,
    "grid.alpha": 0.4,
    "font.size": 11,
})
ORANGE = "#E07B39"
BROWN  = "#6B3F1A"
GOLD   = "#C9A84C"

In [9]:
# Step 1: Data Acquisition and Data Cleaning
print("STEP 1 — DATA ACQUISITION & CLEANING")
print("-"*60)
 
# 1A. Fetch PM2.5 hourly data from Open-Meteo Air Quality API
print("\n[1A] Fetching PM2.5 air quality data from Open-Meteo…")
 
AQ_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
AQ_PARAMS = {
    "latitude":   13.754,
    "longitude":  100.5014,
    "hourly":     "pm2_5",
    "start_date": "2016-01-01",
    "end_date":   "2025-12-31",
    "timezone":   "Asia/Bangkok",
}
 
resp_aq = requests.get(AQ_URL, params=AQ_PARAMS, timeout=60)
resp_aq.raise_for_status()
aq_json = resp_aq.json()
 
df_aq_hourly = pd.DataFrame({
    "datetime": pd.to_datetime(aq_json["hourly"]["time"]),
    "pm2_5":    aq_json["hourly"]["pm2_5"],
})
print(f" Air quality records fetched : {len(df_aq_hourly):,} hourly rows")
 
# 1B. Fetch meteorological hourly data from Open-Meteo Archive API 
print("\n[1B] Fetching meteorological data from Open-Meteo Archive API…")
 
METEO_URL = "https://archive-api.open-meteo.com/v1/archive"
METEO_PARAMS = {
    "latitude":   13.754,
    "longitude":  100.5014,
    "hourly":     "temperature_2m,precipitation,wind_speed_10m,wind_direction_10m",
    "start_date": "2016-01-01",
    "end_date":   "2025-12-31",
    "timezone":   "Asia/Bangkok",
}
 
resp_m = requests.get(METEO_URL, params=METEO_PARAMS, timeout=60)
resp_m.raise_for_status()
m_json = resp_m.json()
 
df_meteo_hourly = pd.DataFrame({
    "datetime":          pd.to_datetime(m_json["hourly"]["time"]),
    "temperature_2m":    m_json["hourly"]["temperature_2m"],
    "precipitation":     m_json["hourly"]["precipitation"],
    "wind_speed_10m":    m_json["hourly"]["wind_speed_10m"],
    "wind_direction_10m":m_json["hourly"]["wind_direction_10m"],
})
print(f" Meteorological records fetched: {len(df_meteo_hourly):,} hourly rows")
 
# 1C. Aggregate from hourly → daily 
print("\n[1C] Aggregating hourly data to daily…")
 
df_aq_hourly["date"] = df_aq_hourly["datetime"].dt.date
 
df_aq_daily = (
    df_aq_hourly
    .groupby("date")
    .agg(pm2_5_mean=("pm2_5", "mean"),
         pm2_5_max =("pm2_5", "max"),
         pm2_5_min =("pm2_5", "min"))
    .reset_index()
)
 
df_meteo_hourly["date"] = df_meteo_hourly["datetime"].dt.date
 
df_meteo_daily = (
    df_meteo_hourly
    .groupby("date")
    .agg(
        temp_mean      =("temperature_2m",     "mean"),
        temp_max       =("temperature_2m",     "max"),
        temp_min       =("temperature_2m",     "min"),
        precip_total   =("precipitation",      "sum"),
        wind_speed_max =("wind_speed_10m",     "max"),
        wind_speed_mean=("wind_speed_10m",     "mean"),
        # dominant wind direction = circular mean via sine/cosine
        _wind_sin      =("wind_direction_10m", lambda x: np.sin(np.deg2rad(x)).mean()),
        _wind_cos      =("wind_direction_10m", lambda x: np.cos(np.deg2rad(x)).mean()),
    )
    .reset_index()
)
 
# Convert circular mean back to degrees [0, 360)
df_meteo_daily["wind_dir_dominant"] = (
    np.rad2deg(np.arctan2(df_meteo_daily["_wind_sin"], df_meteo_daily["_wind_cos"])) % 360
)
df_meteo_daily.drop(columns=["_wind_sin", "_wind_cos"], inplace=True)
 
# 1D. Merge on date 
df = pd.merge(df_aq_daily, df_meteo_daily, on="date", how="inner")
df["date"] = pd.to_datetime(df["date"])
df.sort_values("date", inplace=True)
df.reset_index(drop=True, inplace=True)
print(f" Merged daily dataset shape : {df.shape}")
 
# 1E. Missing value report & imputation
print("\n[1E] Missing value audit:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
miss_report = pd.concat([missing, missing_pct], axis=1, keys=["count", "%"])
print(miss_report[miss_report["count"] > 0].to_string() or "   No missing values found.")
 
# Forward-fill then back-fill (preserves temporal order, avoids leakage)
df.set_index("date", inplace=True)
df = df.ffill().bfill()
df.reset_index(inplace=True)
 
# Sanity check: remove physically impossible PM2.5 values
df.loc[df["pm2_5_mean"] < 0, "pm2_5_mean"] = np.nan
df["pm2_5_mean"] = df["pm2_5_mean"].interpolate(method="linear")
 
print(f"\n Remaining nulls after imputation : {df.isnull().sum().sum()}")
print(f" Final dataset date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f" Final dataset shape: {df.shape}")
 
# Save cleaned dataset 
df.to_csv(f"{OUTPUT_DIR}/pm25_bangkok_clean.csv", index=False)
print(f"\n Cleaned dataset saved: {OUTPUT_DIR}/pm25_bangkok_clean.csv")

STEP 1 — DATA ACQUISITION & CLEANING
------------------------------------------------------------

[1A] Fetching PM2.5 air quality data from Open-Meteo…
 Air quality records fetched : 87,672 hourly rows

[1B] Fetching meteorological data from Open-Meteo Archive API…
 Meteorological records fetched: 87,672 hourly rows

[1C] Aggregating hourly data to daily…
 Merged daily dataset shape : (3653, 11)

[1E] Missing value audit:
            count      %
pm2_5_mean   2407  65.89
pm2_5_max    2407  65.89
pm2_5_min    2407  65.89

 Remaining nulls after imputation : 0
 Final dataset date range: 2016-01-01 → 2025-12-31
 Final dataset shape: (3653, 11)

 Cleaned dataset saved: outputs/pm25_bangkok_clean.csv


In [11]:
# Step 2: EDA
print("STEP 2 — EXPLORATORY DATA ANALYSIS")
print("="*60)
 
# 2A. Summary Statistics 
print("\n[2A] Summary statistics (key columns):")
key_cols = ["pm2_5_mean", "temp_mean", "precip_total", "wind_speed_mean"]
print(df[key_cols].describe().round(2).to_string())
 
# WHO daily PM2.5 guideline = 15 µg/m³
who_limit = 15
exceed_pct = (df["pm2_5_mean"] > who_limit).mean() * 100
print(f"\n Days exceeding WHO 15 µg/m³ guideline: {exceed_pct:.1f}%")
 
# 2B. Full time-series of PM2.5 
print("\n[2B] Plotting full PM2.5 time-series…")
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(df["date"], df["pm2_5_mean"], color=ORANGE, lw=0.7, alpha=0.8, label="Daily PM2.5")
ax.axhline(who_limit, color="red", lw=1.2, ls="--", label="WHO guideline (15 µg/m³)")
ax.set_title("Daily Mean PM2.5 — Bangkok 2016–2025", fontsize=14, fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("PM2.5 (µg/m³)")
ax.legend(); fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/eda_01_timeseries.png")
plt.close()
 
# 2C. Monthly Seasonality (box-plot)
print("[2C] Plotting monthly seasonality…")
df["month"]     = df["date"].dt.month
df["month_name"]= df["date"].dt.strftime("%b")
MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
 
fig, ax = plt.subplots(figsize=(14, 5))
month_data = [df.loc[df["month"]==m, "pm2_5_mean"].values for m in range(1,13)]
bp = ax.boxplot(month_data, patch_artist=True, medianprops=dict(color="white", lw=2),
                whiskerprops=dict(color=BROWN), capprops=dict(color=BROWN),
                flierprops=dict(marker=".", color=ORANGE, alpha=0.4, ms=4))
colors = [ORANGE if m in [1,2,3,12] else GOLD if m in [4,11] else "#A0A0A0" for m in range(1,13)]
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c)
ax.axhline(who_limit, color="red", lw=1.2, ls="--", label="WHO 15 µg/m³")
ax.set_xticklabels(MONTHS); ax.set_xlabel("Month"); ax.set_ylabel("PM2.5 (µg/m³)")
ax.set_title("Monthly PM2.5 Distribution — Bangkok (orange = dry/haze season)", fontsize=13, fontweight="bold")
ax.legend(); fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/eda_02_monthly_boxplot.png")
plt.close()
 
# 2D. Year-over-year trend 
print("[2D] Plotting year-over-year trend…")
df["year"] = df["date"].dt.year
yearly = df.groupby("year")["pm2_5_mean"].agg(["mean","median","max"]).reset_index()
 
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(yearly["year"], yearly["mean"], color=ORANGE, alpha=0.8, label="Annual mean")
ax.plot(yearly["year"], yearly["median"], "o-", color=BROWN, lw=2, label="Annual median")
ax.axhline(who_limit, color="red", lw=1.2, ls="--", label="WHO 15 µg/m³")
ax.set_title("Year-over-Year PM2.5 — Bangkok", fontsize=13, fontweight="bold")
ax.set_xlabel("Year"); ax.set_ylabel("PM2.5 (µg/m³)")
ax.legend(); fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/eda_03_yearly_trend.png")
plt.close()
 
# 2E. Correlation heatmap 
print("[2E] Plotting correlation heatmap…")
corr_cols = ["pm2_5_mean","temp_mean","temp_max","precip_total",
             "wind_speed_mean","wind_speed_max","wind_dir_dominant"]
corr_matrix = df[corr_cols].corr()
 
fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f",
            cmap="RdYlGn_r", center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax)
ax.set_title("Pearson Correlation Matrix", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/eda_04_correlation.png")
plt.close()
 
# 2F. Scatter: PM2.5 vs key drivers
print("[2F] Plotting PM2.5 vs meteorological drivers…")
drivers = [("wind_speed_mean","Mean Wind Speed (km/h)"),
           ("precip_total",   "Daily Precipitation (mm)"),
           ("temp_mean",      "Mean Temperature (°C)")]
 
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (col, xlabel) in zip(axes, drivers):
    ax.scatter(df[col], df["pm2_5_mean"], alpha=0.15, s=6, color=ORANGE)
    # Regression line
    m, b, r, p, _ = stats.linregress(df[col].dropna(), df.loc[df[col].notna(), "pm2_5_mean"])
    x_range = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(x_range, m*x_range + b, color=BROWN, lw=2, label=f"r={r:.2f}")
    ax.set_xlabel(xlabel); ax.set_ylabel("PM2.5 (µg/m³)")
    ax.set_title(f"PM2.5 vs {xlabel.split('(')[0].strip()}", fontsize=11)
    ax.legend()
fig.suptitle("PM2.5 vs Meteorological Drivers", fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/eda_05_scatter_drivers.png", bbox_inches="tight")
plt.close()
 
# 2G. PM2.5 distribution 
print("[2G] Plotting PM2.5 distribution…")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
 
axes[0].hist(df["pm2_5_mean"], bins=60, color=ORANGE, edgecolor="white", alpha=0.85)
axes[0].axvline(df["pm2_5_mean"].mean(), color=BROWN, lw=2, ls="--", label=f"Mean={df['pm2_5_mean'].mean():.1f}")
axes[0].axvline(who_limit, color="red", lw=2, ls=":", label="WHO 15 µg/m³")
axes[0].set_xlabel("PM2.5 (µg/m³)"); axes[0].set_ylabel("Frequency")
axes[0].set_title("PM2.5 Frequency Distribution")
axes[0].legend()
 
axes[1].hist(np.log1p(df["pm2_5_mean"]), bins=60, color=GOLD, edgecolor="white", alpha=0.85)
axes[1].set_xlabel("log(1 + PM2.5)"); axes[1].set_ylabel("Frequency")
axes[1].set_title("PM2.5 Log-Transformed Distribution")
 
fig.suptitle("PM2.5 Distribution Analysis", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/eda_06_distribution.png")
plt.close()
 
# 2H. Average daily profile of PM2.5 by month (heatmap) 
print("[2H] Plotting monthly-year heatmap…")
pivot = df.pivot_table(values="pm2_5_mean", index="year", columns="month", aggfunc="mean")
pivot.columns = MONTHS
 
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlOrRd",
            linewidths=0.5, ax=ax, cbar_kws={"label":"PM2.5 (µg/m³)"})
ax.set_title("Average Monthly PM2.5 by Year — Bangkok", fontsize=13, fontweight="bold")
ax.set_xlabel("Month"); ax.set_ylabel("Year")
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/eda_07_monthly_year_heatmap.png")
plt.close()
 
print("\ EDA plots saved to outputs/ folder")

STEP 2 — EXPLORATORY DATA ANALYSIS

[2A] Summary statistics (key columns):
       pm2_5_mean  temp_mean  precip_total  wind_speed_mean
count     3653.00    3653.00       3653.00          3653.00
mean        19.27      28.07          4.27             8.72
std         19.44       1.75          7.06             2.78
min          0.72      18.03          0.00             1.80
25%         12.88      27.17          0.00             6.67
50%         12.88      28.06          1.10             8.42
75%         13.35      29.03          5.80            10.45
max        276.15      33.18         80.70            22.37

 Days exceeding WHO 15 µg/m³ guideline: 23.1%

[2B] Plotting full PM2.5 time-series…
[2C] Plotting monthly seasonality…
[2D] Plotting year-over-year trend…
[2E] Plotting correlation heatmap…
[2F] Plotting PM2.5 vs meteorological drivers…
[2G] Plotting PM2.5 distribution…
[2H] Plotting monthly-year heatmap…
\ EDA plots saved to outputs/ folder


In [13]:
#Step 3: Feature Engineering
print("STEP 3 — FEATURE ENGINEERING")
print("="*60)
 
# Work on a copy to keep the clean dataset intact
dff = df.copy().sort_values("date").reset_index(drop=True)
 
# 3A. Lag Features (avoid leakage: only look backward)
print("\n[3A] Creating lag features for PM2.5…")
for lag in [1, 2, 3, 7, 14]:
    dff[f"pm2_5_lag_{lag}d"] = dff["pm2_5_mean"].shift(lag)
 
# 3B. Rolling Window Statistics 
print("[3B] Creating rolling-window statistics…")
for window in [3, 7, 14, 30]:
    # Using min_periods=1 so rows near the start are not dropped
    dff[f"pm2_5_roll_mean_{window}d"] = (
        dff["pm2_5_mean"].shift(1).rolling(window, min_periods=1).mean()
    )
    dff[f"pm2_5_roll_std_{window}d"] = (
        dff["pm2_5_mean"].shift(1).rolling(window, min_periods=1).std()
    )
 
# Rolling mean for wind speed (captures sustained calm periods)
dff["wind_speed_roll_7d"] = (
    dff["wind_speed_mean"].shift(1).rolling(7, min_periods=1).mean()
)
 
# 3C. Wind Direction Decomposition (sine/cosine) 
print("[3C] Decomposing wind direction into sin/cos components…")
wind_rad = np.deg2rad(dff["wind_dir_dominant"])
dff["wind_sin"] = np.sin(wind_rad)
dff["wind_cos"] = np.cos(wind_rad)
 
# 3D. Calendar / Time Features 
print("[3D] Adding calendar features…")
dff["day_of_year"]    = dff["date"].dt.dayofyear
dff["day_of_week"]    = dff["date"].dt.dayofweek          # 0=Mon, 1=Tues, 2=Wed, 3=Thurs, 4=Fri, 5=Sat, 6=Sun
dff["is_weekend"]     = (dff["day_of_week"] >= 5).astype(int)
dff["week_of_year"]   = dff["date"].dt.isocalendar().week.astype(int)
dff["quarter"]        = dff["date"].dt.quarter
 
# Cyclical encoding of day-of-year (captures seasonality smoothly)
dff["doy_sin"] = np.sin(2 * np.pi * dff["day_of_year"] / 365.25)
dff["doy_cos"] = np.cos(2 * np.pi * dff["day_of_year"] / 365.25)
 
# Cyclical encoding of month
dff["month_sin"] = np.sin(2 * np.pi * dff["month"] / 12)
dff["month_cos"] = np.cos(2 * np.pi * dff["month"] / 12)
 
# 3E. Haze Season Flag 
print("[3E] Adding domain-knowledge feature: haze season flag…")
# Bangkok haze season = Jan, Feb, Mar (crop burning + temperature inversion)
dff["is_haze_season"] = dff["month"].isin([1, 2, 3]).astype(int)
 
# 3F. Precipitation Flag (dry vs wet day) 
dff["is_dry_day"] = (dff["precip_total"] < 1.0).astype(int)   # <1mm = dry
dff["is_heavy_rain"] = (dff["precip_total"] > 20.0).astype(int) # >20mm = wet
 
# 3G. Log-transform of PM2.5 (useful as alternative target)
dff["pm2_5_log"] = np.log1p(dff["pm2_5_mean"])
 
# 3H. Target variable (t+1 day ahead) 
print("[3H] Creating prediction target: next-day PM2.5…")
dff["pm2_5_target_t1"] = dff["pm2_5_mean"].shift(-1)   # predict tomorrow
# (last row will be NaN — drop only when training, not here)
 
# Drop rows with NaN introduced by lag at the very start 
MIN_LAG = 14  # longest lag used
dff_model_ready = dff.iloc[MIN_LAG:].copy()
print(f" Rows available after lag warm-up (first {MIN_LAG} days dropped): {len(dff_model_ready):,}")
 
# 3I. Feature summary 
feature_cols = [c for c in dff_model_ready.columns
                if c not in ["date","pm2_5_max","pm2_5_min","month","year",
                              "month_name","day_of_year"]]
print(f"\n   Total feature columns ready for modelling: {len(feature_cols)}")
print(" Feature list:")
for fc in sorted(feature_cols):
    print(f"• {fc}")
 
# 3J. Visualise engineered features 
print("\n[3J] Plotting engineered feature illustration…")
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
sample = dff_model_ready[(dff_model_ready["year"] >= 2023) &
                          (dff_model_ready["year"] <= 2024)]
 
axes[0].plot(sample["date"], sample["pm2_5_mean"],      color=ORANGE, lw=0.8, label="Daily PM2.5", alpha=0.7)
axes[0].plot(sample["date"], sample["pm2_5_roll_mean_7d"],  color=BROWN, lw=2,   label="7-day rolling mean")
axes[0].plot(sample["date"], sample["pm2_5_roll_mean_30d"], color="gray", lw=1.5, ls="--", label="30-day rolling mean")
axes[0].set_ylabel("PM2.5 (µg/m³)"); axes[0].legend(fontsize=9)
axes[0].set_title("PM2.5 with Rolling Averages (2023–2024)", fontweight="bold")
 
axes[1].plot(sample["date"], sample["pm2_5_lag_1d"], color=GOLD,  lw=0.9, label="Lag 1d")
axes[1].plot(sample["date"], sample["pm2_5_lag_7d"], color=BROWN, lw=0.9, label="Lag 7d", alpha=0.7)
axes[1].plot(sample["date"], sample["pm2_5_mean"],   color=ORANGE, lw=0.8, alpha=0.5, label="Actual")
axes[1].set_ylabel("PM2.5 (µg/m³)"); axes[1].legend(fontsize=9)
axes[1].set_title("Lag Features vs Actual PM2.5", fontweight="bold")
 
axes[2].fill_between(sample["date"], sample["wind_sin"], 0,
                     alpha=0.5, color=ORANGE, label="Wind sin")
axes[2].fill_between(sample["date"], sample["wind_cos"], 0,
                     alpha=0.5, color=BROWN,  label="Wind cos")
axes[2].set_ylabel("Amplitude"); axes[2].set_xlabel("Date")
axes[2].legend(fontsize=9)
axes[2].set_title("Wind Direction Decomposed (sin/cos)", fontweight="bold")
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
 
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/feature_engineering_preview.png")
plt.close()
 
# Save feature-engineered dataset 
dff_model_ready.to_csv(f"{OUTPUT_DIR}/pm25_bangkok_features.csv", index=False)
print(f"\n Feature-engineered dataset saved → {OUTPUT_DIR}/pm25_bangkok_features.csv")

STEP 3 — FEATURE ENGINEERING

[3A] Creating lag features for PM2.5…
[3B] Creating rolling-window statistics…
[3C] Decomposing wind direction into sin/cos components…
[3D] Adding calendar features…
[3E] Adding domain-knowledge feature: haze season flag…
[3H] Creating prediction target: next-day PM2.5…
 Rows available after lag warm-up (first 14 days dropped): 3,639

   Total feature columns ready for modelling: 37
 Feature list:
• day_of_week
• doy_cos
• doy_sin
• is_dry_day
• is_haze_season
• is_heavy_rain
• is_weekend
• month_cos
• month_sin
• pm2_5_lag_14d
• pm2_5_lag_1d
• pm2_5_lag_2d
• pm2_5_lag_3d
• pm2_5_lag_7d
• pm2_5_log
• pm2_5_mean
• pm2_5_roll_mean_14d
• pm2_5_roll_mean_30d
• pm2_5_roll_mean_3d
• pm2_5_roll_mean_7d
• pm2_5_roll_std_14d
• pm2_5_roll_std_30d
• pm2_5_roll_std_3d
• pm2_5_roll_std_7d
• pm2_5_target_t1
• precip_total
• quarter
• temp_max
• temp_mean
• temp_min
• week_of_year
• wind_cos
• wind_dir_dominant
• wind_sin
• wind_speed_max
• wind_speed_mean
• wind_speed_